# 실습 3 — 소실점으로 자유도를 줄여 만드는 Around-View

## 오늘의 목표

실습 2에서 확인한 호모그래피의 한계:

> **8-DoF → 대응점 4개 이상, non-collinear, 전 영역 분산 필요**

차량 주변 360°에 정밀 기준점을 넓게 깔기는 현실적으로 어렵다.
근거리 한 곳에 몰린 4점으로 추정하면 원거리에서 오차가 폭증한다.

**해법**: 지면의 평행선에서 얻은 **소실점 2개**가 호모그래피의
자유도를 8 → 4 (또는 3, 2) 로 줄여준다. 대응점은 2개, 심지어 1개로 충분해진다.

## 진행

| 단계 | 내용 | 빈칸 |
|---|---|---|
| 0 | 합성 리그 생성 · 영상 확인 | |
| 1 | 선분 검출 → RANSAC 소실점 | **①** |
| 2 | 소실점 → 회전 R | **②** |
| 3 | 대응점 → 평행이동 t | **③** |
| 4 | 이산 모호성(8중) 해소 | |
| 5 | DLT 와 정량 비교 | |
| 6 | Around-View 합성 | |

빈칸에서 막히면 `from solutions import ...` 로 가져다 쓰고 진행할 것.

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
import vp_avm as V
import synth_rig as S

np.set_printoptions(precision=3, suppress=True)
def show(img, title="", figsize=(9,6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim==3 else img, cmap="gray")
    plt.title(title); plt.axis("off"); plt.show()

print("OpenCV", cv2.__version__)

## 0. 합성 리그

외부 데이터 없이 코드로 장면을 만든다. 실데이터로 바꿀 때는 `synth_rig.py` 만 교체하면 된다.

- 세계좌표: 지면 Z=0, **+X 전방 / +Y 좌측 / +Z 위**
- 카메라: **x 우 / y 아래 / z 광축** (OpenCV 관례)
- 4카메라(front/left/rear/right), 높이 ~1.0m, 하향틸트 ~40°, FOV 110°

In [ ]:
tex  = S.make_ground_texture()
cams = S.make_rig()
imgs = {n: S.render(c, tex, seed=i) for i,(n,c) in enumerate(cams.items())}

fig, ax = plt.subplots(2,2, figsize=(13,8))
for a,(n,im) in zip(ax.ravel(), imgs.items()):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(n); a.axis("off")
plt.tight_layout(); plt.show()

cam = cams["front"]
print("K =\n", cam["K"])
print("정답 C =", cam["C"], " yaw =", cam["yaw"], " pitch = %.1f" % cam["pitch"])

## 1. 선분 → 소실점  【빈칸 ①】

### 원리

세계에서 평행한 직선들은 영상에서 **한 점**에서 만난다. 그 점이 소실점 $v$.

영상 직선 $l$ 위에 $v$ 가 있으므로 $l^\top v = 0$.
같은 방향의 직선 $M$ 개를 모으면

$$\underbrace{\begin{bmatrix}l_1^\top\\ \vdots \\ l_M^\top\end{bmatrix}}_{L}\,v = 0
\quad\Longrightarrow\quad \min_{\|v\|=1}\|Lv\|$$

→ $L$ 의 **최소 특이값에 대응하는 우특이벡터**.

### 중요

입력은 "이 선들이 세계에서 서로 평행하다"는 사실 **뿐**이다.
선의 실제 좌표·간격·원점은 몰라도 된다. 주차선·차선·연석·타일 줄눈이 전부 재료가 된다.

In [ ]:
def refine_vp(lines):
    """
    lines : (M,3) 정규화된 직선들 (같은 세계 방향)
    반환  : (3,) 소실점 (동차좌표, 단위노름)
    """
    # ===== 빈칸 ① =====
    # 힌트: np.linalg.svd(lines) 의 Vt 마지막 행
    raise NotImplementedError
    # ==================

# --- 자가 검증 게이트 ---
try:
    _K = cams["front"]["K"]; _R = cams["front"]["R"]
    _v = _K @ _R[:,0]; _v /= np.linalg.norm(_v)          # 정답 소실점 (세계 +X)
    _rng = np.random.default_rng(0)
    _pts = _rng.uniform(-500,500,(12,2))
    _L = np.cross(np.c_[_pts, np.ones(12)], _v[None,:])   # v 를 지나는 직선들
    _L /= np.linalg.norm(_L[:,:2],axis=1,keepdims=True)
    _est = refine_vp(_L); _est *= np.sign(_est @ _v)
    err = np.degrees(np.arccos(np.clip(_est @ _v,-1,1)))
    print(f"소실점 각도 오차 {err:.4f}deg  ->", "PASS" if err < 1e-3 else "FAIL")
except NotImplementedError:
    from solutions import refine_vp
    print("solutions.refine_vp 사용")

In [ ]:
# RANSAC + greedy 로 지배적 소실점들을 순차 검출 (vp_avm 제공)
gray = cv2.cvtColor(imgs["front"], cv2.COLOR_BGR2GRAY)
segs = V.detect_segments(gray, min_len=45)
vps  = V.estimate_vps(segs, n_vp=3, tol_deg=1.5, refine_fn=refine_vp)
print(f"선분 {len(segs)}개 -> 소실점 {len(vps)}개, 인라이어 {[int(m.sum()) for _,m in vps]}")

vis = imgs["front"].copy()
for (vp,mask),col in zip(vps[:2], [(0,255,255),(255,120,0)]):
    for s in segs[mask]:
        cv2.line(vis,(int(s[0]),int(s[1])),(int(s[2]),int(s[3])),col,2)
show(vis, "Direction clusters (yellow / blue)")

### 직교 방향쌍 고르기

검출된 소실점 중 **세계에서 서로 직교**하는 쌍을 골라야 한다.
$K$ 를 알면 $d_i = K^{-1}v_i$ 가 그 방향의 단위벡터이므로

$$d_1 \cdot d_2 \approx 0$$

인 쌍을 고르면 된다. (참고: 서로 직교하는 소실점 **3개**가 있으면
$K$ 자체를 복원할 수도 있다 — Zhang 캘리브레이션의 $\omega$ 제약과 같은 형태)

In [ ]:
v_a, v_b, ang = V.pick_orthogonal_pair(vps, cam["K"])
print(f"선택된 쌍의 직교각 = {ang:.2f}deg   (90도에 가까울수록 좋음)")

## 2. 소실점 → 회전 R  【빈칸 ②】

### 왜 이게 되는가

지면 $Z=0$ 위의 점 $(X,Y,0,1)$ 에 대해

$$x \sim K[R\,|\,t]\begin{bmatrix}X\\Y\\0\\1\end{bmatrix}
 = K[\,r_1\ r_2\ t\,]\begin{bmatrix}X\\Y\\1\end{bmatrix}
 \;\Rightarrow\; \boxed{H_{g\to i}=K[\,r_1\ r_2\ t\,]}$$

여기에 **무한원점**을 넣어보자. 세계 $+X$ 방향의 무한원점은 $(1,0,0,0)$ 이므로

$$v_X \sim K[R|t](1,0,0,0)^\top = K r_1, \qquad v_Y \sim K r_2$$

즉 **소실점 2개 = $H$ 의 앞 두 열**. $r_1, r_2$ 는 단위벡터이므로 정규화로 스케일까지 결정된다.

$$r_1=\frac{K^{-1}v_X}{\|K^{-1}v_X\|},\quad
  r_2=\frac{K^{-1}v_Y}{\|K^{-1}v_Y\|},\quad r_3=r_1\times r_2$$

잡음 때문에 $r_1 \perp r_2$ 가 정확히 성립하지 않으므로 SVD 로 $SO(3)$ 에 투영한다.

In [ ]:
def R_from_vps(v1, v2, K):
    """v1 ~ K r1, v2 ~ K r2 -> 회전행렬 R (world->cam)"""
    # ===== 빈칸 ② =====
    # 1) Ki = inv(K);  r1 = normalize(Ki@v1);  r2 = normalize(Ki@v2)
    # 2) M = [r1, r2, cross(r1,r2)]  (열로 쌓기)
    # 3) U,_,Vt = svd(M);  R = U@Vt;  det<0 이면 U[:,-1] *= -1 후 재계산
    raise NotImplementedError
    # ==================

try:
    _R = R_from_vps(cam["K"]@cam["R"][:,0], cam["K"]@cam["R"][:,1], cam["K"])
    d = np.degrees(np.arccos(np.clip((np.trace(_R@cam["R"].T)-1)/2,-1,1)))
    print(f"회전 오차 {d:.4f}deg  ->", "PASS" if d < 1e-3 else "FAIL")
except NotImplementedError:
    from solutions import R_from_vps
    print("solutions.R_from_vps 사용")

## 3. 대응점 → 평행이동 t  【빈칸 ③】

$R$ 이 정해졌으니 남은 미지수는 $t \in \mathbb{R}^3$ 뿐이다.

정규화 광선 $m = K^{-1}x$ 는 카메라 좌표의 점 $r_1X + r_2Y + t$ 와 **평행**하므로

$$m \times (r_1X + r_2Y + t) = 0
\;\Longleftrightarrow\;
[m]_\times\, t = -[m]_\times (r_1X + r_2Y)$$

$[m]_\times$ 는 rank 2 → **점 1개당 독립식 2개**. 미지수 3개이므로 **최소 2점**.

In [ ]:
def solve_t(R, K, img_pts, gnd_pts):
    """R,K 기지. 대응점 2개 이상으로 t 를 최소제곱 추정"""
    # ===== 빈칸 ③ =====
    # for (u,v),(X,Y) in zip(img_pts, gnd_pts):
    #     m = normalize(inv(K) @ [u,v,1]);  S = skew(m)
    #     A += [S];  b += [-S @ (R[:,0]*X + R[:,1]*Y)]
    # lstsq(vstack(A), concatenate(b))
    raise NotImplementedError
    # ==================

try:
    g,i_ = S.make_correspondences(cam,"cluster",noise=0.0)
    _t = solve_t(cam["R"], cam["K"], i_[:2], g[:2])
    e = np.linalg.norm(_t - cam["t"])
    print(f"t 오차 {e:.5f} m  ->", "PASS" if e < 1e-6 else "FAIL")
except NotImplementedError:
    from solutions import solve_t
    print("solutions.solve_t 사용")

## 4. 자유도 장부 정리 + 8중 이산 모호성

### 자유도가 어떻게 줄어드는가

| 아는 것 | 남은 미지수 | 연속 DoF | 필요 대응점 |
|---|---|---|---|
| — (일반 DLT) | $H$ 전체 | **8** | **4** |
| 소실점 2개 (K 몰라도 됨) | $\lambda_1,\lambda_2,h_3$ − 전체스케일 | **4** | **2** |
| 소실점 2개 + $K$ | $t$ | **3** | **2** |
| 위 + 장착높이 $h$ | $C_x, C_y$ | **2** | **1** |

### 남는 것은 **이산** 모호성 8개

소실점은 방향의 **부호**를 구별하지 못하고($v$ 와 $-v$ 가 같은 점),
어느 쪽이 $+X$ 인지도 모른다.

$$2\ (\text{라벨 교환}) \times 2\ (v_1\text{ 부호}) \times 2\ (v_2\text{ 부호}) = 8$$

걸러내는 순서:

1. **지면법선**: $r_3 = Re_3$ = 카메라에서 본 세계 $+Z$. 카메라 $y$ 축이 아래를 향하므로 $r_3[1] < 0$ → **8 → 4**
2. **cheirality**: 카메라 높이 범위, 관측점 depth > 0
3. **남은 4중** = 세계좌표를 $Z$축 기준 90°씩 돌린 것. 대응점 재투영오차로 깨거나, 리그 장착 방위(prior)로 지정

> ⚠️ **함정**: 대응점 배치가 90° 회전 대칭(정사각형 4점)이면 3번이 깨지지 않는다.
> ⚠️ **함정**: 1점 + 높이 모드는 식 수 = 미지수라 잔차가 항상 0 → prior 없이는 절대 구분 불가.

In [ ]:
g_cl, i_cl = S.make_correspondences(cam, "cluster", noise=0.7, seed=1)

res2 = V.estimate_H_vp(cam["K"], v_a, v_b, i_cl[:2], g_cl[:2],
                       yaw_prior_deg=cam["yaw"], refine=(R_from_vps, solve_t))
res1 = V.estimate_H_vp(cam["K"], v_a, v_b, i_cl[:1], g_cl[:1],
                       cam_height=cam["C"][2], yaw_prior_deg=cam["yaw"],
                       refine=(R_from_vps, solve_t))

for tag,r in [("VP+2pt",res2), ("VP+1pt+h",res1)]:
    print(f"{tag:10s} 생존후보 {r['n_candidates']}개 | 라벨교환={r['label_swapped']} "
          f"부호={r['signs']} | C={r['C']}  (정답 {cam['C']})")

## 5. DLT 와 정량 비교

평가 지표: **지면 역투영 RMSE** — 시야 안 격자점을 영상에 투영한 뒤,
추정 $H^{-1}$ 로 다시 지면으로 되돌렸을 때의 위치 오차(미터).

### 왜 근거리 4점 DLT 가 무너지는가

카메라 높이 $h$, 틸트 오차 $\delta\theta$ 일 때 거리 $d$ 지점의 지면 오차는

$$\Delta \approx \frac{d^2}{h}\,\delta\theta$$

**거리의 제곱**으로 커진다. 근거리 한 뼘 패치로는 틸트(=방향)가 거의 관측되지 않으므로
$\delta\theta$ 가 크고, 원거리에서 폭발한다.

소실점은 **무한원거리의 방향 정보**를 직접 주므로 $\delta\theta$ 를 근본적으로 잡는다.
→ **소실점 = 방향(원거리), 대응점 = 위치·축척(근거리)** 의 역할 분담.

In [ ]:
rows=[]
for name,c in cams.items():
    gray = cv2.cvtColor(imgs[name], cv2.COLOR_BGR2GRAY)
    sg   = V.detect_segments(gray, min_len=45)
    vp   = V.estimate_vps(sg, n_vp=3, tol_deg=1.5, refine_fn=refine_vp)
    va, vb, an = V.pick_orthogonal_pair(vp, c["K"])
    gw,iw = S.make_correspondences(c,"wide",   noise=0.7, seed=2)
    gc,ic = S.make_correspondences(c,"cluster",noise=0.7, seed=1)

    H = {}
    H["A DLT wide-4"]    = V.estimate_H_dlt(iw, gw) if len(gw)>=4 else None
    H["B DLT cluster-4"] = V.estimate_H_dlt(ic, gc) if len(gc)>=4 else None
    H["C VP + 2pt"]      = V.estimate_H_vp(c["K"],va,vb,ic[:2],gc[:2],
                             yaw_prior_deg=c["yaw"], refine=(R_from_vps,solve_t))["H"]
    H["D VP + 1pt + h"]  = V.estimate_H_vp(c["K"],va,vb,ic[:1],gc[:1],
                             cam_height=c["C"][2], yaw_prior_deg=c["yaw"],
                             refine=(R_from_vps,solve_t))["H"]
    rows.append((name, {k:(S.ground_rmse(v,c) if v is not None else np.nan)
                        for k,v in H.items()}, H, an))

keys = ["A DLT wide-4","B DLT cluster-4","C VP + 2pt","D VP + 1pt + h"]
print(f"{'cam':<7}" + "".join(f"{k:>18}" for k in keys) + f"{'VP각':>8}")
for n,e,_,an in rows:
    print(f"{n:<7}" + "".join(f"{e[k]:>18.3f}" for k in keys) + f"{an:>7.1f}°")
print("-"*85)
print(f"{'평균':<7}" + "".join(f"{np.nanmean([e[k] for _,e,_,_ in rows]):>18.3f}" for k in keys))

## 6. Around-View 합성

카메라별 $H_{g\to i}$ 가 나왔으니 BEV 평면으로 워핑하고 거리변환 페더 블렌딩으로 합친다.

$$H_{\text{bev}\to i} = H_{g\to i}\, M_{\text{bev}}^{-1}$$

In [ ]:
def avm(key):
    ims = [imgs[n] for n,_,H,_ in rows if H.get(key) is not None]
    Hs  = [H[key]  for _,_,H,_ in rows if H.get(key) is not None]
    return V.render_around_view(ims, Hs, (-9,9), (-9,9), ppm=45, feather=61)[0]

gt = V.render_around_view([imgs[n] for n in cams], [cams[n]["H"] for n in cams],
                          (-9,9), (-9,9), ppm=45, feather=61)[0]

fig,ax = plt.subplots(1,3, figsize=(16,5.6))
for a,(im,t) in zip(ax, [(gt,"Ground truth"),
                         (avm("B DLT cluster-4"),"B: DLT 4pt (near cluster)"),
                         (avm("C VP + 2pt"),"C: VP + 2pt")]):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 7. 정리와 남은 한계

### 얻은 것

- 소실점 2개 = $H$ 의 앞 두 열 → **8-DoF → 4/3/2-DoF**
- 대응점 **4개 → 2개 (또는 1개)**
- 소실점 입력은 "평행하다"는 사실뿐 — 좌표 측량 불필요
- 근거리 패치만으로도 원거리 정확도 유지

### 남은 한계 (다음 단계 = 통합 최적화)

- 지면 비평면성(경사·요철) → 소실점 자체가 흔들림
- 카메라 간 이음매 불일치 → 인접 카메라 공통 특징점으로 joint refine
- 소실점은 **방향만** 고정. 축척·원점은 여전히 대응점 의존
- 이산 모호성은 prior 또는 비대칭 대응점 배치로 반드시 깨야 함

### 실데이터 전환 체크리스트

1. 입력 좌표는 **왜곡 보정 후** 픽셀 (어안: `cv2.fisheye.undistortPoints`)
2. `yaw_prior_deg` 에 리그 설계 장착 방위 입력
3. 소실점 검출 실패 시 `tol_deg` 완화(1.5→3.0), `min_len` 하향
4. 평행선 재료 확보: 주차선 / 차선 / 연석 / 타일 줄눈